# 🔬 10주차 실습 노트북
## 내가 직접 AI 뇌를 설계한다 — MLP 구조 실험

---

**📌 오늘의 핵심 질문**
> **"층이 많을수록, 뉴런이 많을수록 무조건 좋아질까?"**

모델 A · B · C 세 가지 구조를 직접 돌려보고, 스스로 답을 찾아봅니다.

| 조건 | 값 |
|------|----|
| 데이터 | 당뇨 예측 데이터 (9주차 재사용) |
| 손실함수 | Binary Crossentropy (고정) |
| 학습률 | 0.01 (고정) |
| 에포크 | 100 (고정) |
| **바꾸는 것** | **층 수 · 뉴런 수만!** |

---
**⚠️ 실행 전 확인**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → GPU 선택

---
## ⚙️ STEP 0 — 라이브러리 설치 및 설정

구글 코랩에서 처음 실행할 때 한 번만 실행하면 됩니다.

In [ ]:
# 필요한 라이브러리 설치 (코랩 환경)
# !pip install tensorflow scikit-learn pandas matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (코랩)
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], 
               capture_output=True)
matplotlib.rc('font', family='NanumGothic')
matplotlib.rc('axes', unicode_minus=False)
plt.rcParams['figure.dpi'] = 100

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 재현성을 위한 시드 고정
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow 버전: {tf.__version__}')
print('라이브러리 로드 완료! ✅')

---
## 📂 STEP 1 — 데이터 준비

9주차에 사용한 당뇨 예측 데이터를 그대로 재사용합니다.  
데이터 설명은 생략하고 **바로 구조 실험**에 집중합니다!

| 특성 | 설명 |
|------|------|
| Pregnancies | 임신 횟수 |
| Glucose | 혈당 수치 |
| BloodPressure | 혈압 |
| SkinThickness | 피부 두께 |
| Insulin | 인슐린 수치 |
| BMI | 체질량지수 |
| DiabetesPedigreeFunction | 유전적 기능 |
| Age | 나이 |
| **Outcome** | **당뇨 여부 (0: 정상, 1: 당뇨)** |

In [ ]:
# 데이터 불러오기
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.csv'
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)

print('데이터 크기:', df.shape)
print('\n처음 5행:')
df.head()

In [ ]:
# 특성(X)과 정답(y) 분리
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

# 훈련 / 테스트 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ⭐ 정규화 (StandardScaler: 평균 0, 표준편차 1)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'훈련 데이터: {X_train.shape[0]}명  ({X_train.shape[1]}개 특성)')
print(f'테스트 데이터: {X_test.shape[0]}명')
print(f'당뇨 비율: {y_train.mean():.1%} (훈련) / {y_test.mean():.1%} (테스트)')
print('\n정규화 완료! ✅ — 오늘은 구조 실험에 집중합니다')

---
## 🔹 STEP 2 — 모델 A 실행 (얕고 좁음)

가장 단순한 구조입니다. 이 정도로도 충분할까요?

```
입력(8) → Dense(4) → Dense(1)
파라미터 수: 8×4 + 4(편향) + 4×1 + 1(편향) = 37개
```

**💭 실행 전 예상**: val_accuracy가 몇 % 나올 것 같나요?

In [ ]:
# ── 모델 A 구성 ──────────────────────────────────
model_A = Sequential([
    Dense(4, activation='relu', input_shape=(8,)),  # 은닉층 1개
    Dense(1, activation='sigmoid')                  # 출력층
], name='model_A')

model_A.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('── 모델 A 구조 ──')
model_A.summary()

In [ ]:
# 모델 A 학습
history_A = model_A.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0  # 로그 출력 끄기
)

# 최종 성능 출력
final_loss_A    = history_A.history['loss'][-1]
final_val_acc_A = history_A.history['val_accuracy'][-1]
final_train_acc_A = history_A.history['accuracy'][-1]

print('='*40)
print('모델 A 학습 완료!')
print(f'  파라미터 수      : 37개')
print(f'  최종 훈련 정확도  : {final_train_acc_A:.1%}')
print(f'  최종 검증 정확도  : {final_val_acc_A:.1%}')
print(f'  최종 손실        : {final_loss_A:.4f}')
print('='*40)

---
## 🔷 STEP 3 — 모델 B 실행 (기본형 ⭐)

은닉층이 2개인 기본형입니다. 오늘의 **기준 모델**입니다.

```
입력(8) → Dense(16) → Dense(8) → Dense(1)
파라미터 수: (8×16+16) + (16×8+8) + (8×1+1) = 289개
```

**💭 A와 비교**: 층 1개 추가, 뉴런 수 증가 → 정확도가 얼마나 달라질까요?

In [ ]:
# ── 모델 B 구성 ──────────────────────────────────
model_B = Sequential([
    Dense(16, activation='relu', input_shape=(8,)),  # 은닉층 1
    Dense(8,  activation='relu'),                    # 은닉층 2
    Dense(1,  activation='sigmoid')                  # 출력층
], name='model_B')

model_B.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('── 모델 B 구조 ──')
model_B.summary()

In [ ]:
# 모델 B 학습
history_B = model_B.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

final_loss_B    = history_B.history['loss'][-1]
final_val_acc_B = history_B.history['val_accuracy'][-1]
final_train_acc_B = history_B.history['accuracy'][-1]

print('='*40)
print('모델 B 학습 완료!')
print(f'  파라미터 수      : 289개')
print(f'  최종 훈련 정확도  : {final_train_acc_B:.1%}')
print(f'  최종 검증 정확도  : {final_val_acc_B:.1%}')
print(f'  최종 손실        : {final_loss_B:.4f}')
print('='*40)
print(f'\n🔍 A vs B 검증 정확도 차이: {(final_val_acc_B - final_val_acc_A):+.1%}')

---
## 🔵 STEP 4 — 모델 C 실행 (깊고 넓음)

은닉층 3개, 뉴런이 많은 구조입니다. 파라미터가 A의 81배나 됩니다.

```
입력(8) → Dense(64) → Dense(32) → Dense(16) → Dense(1)
파라미터 수: (8×64+64) + (64×32+32) + (32×16+16) + (16×1+1) = 3,025개
```

**💭 예상**: C가 B보다 항상 좋을까요? 아니라면 왜일까요?

In [ ]:
# ── 모델 C 구성 ──────────────────────────────────
model_C = Sequential([
    Dense(64, activation='relu', input_shape=(8,)),  # 은닉층 1
    Dense(32, activation='relu'),                    # 은닉층 2
    Dense(16, activation='relu'),                    # 은닉층 3
    Dense(1,  activation='sigmoid')                  # 출력층
], name='model_C')

model_C.compile(
    optimizer=Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print('── 모델 C 구조 ──')
model_C.summary()

In [ ]:
# 모델 C 학습
history_C = model_C.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

final_loss_C    = history_C.history['loss'][-1]
final_val_acc_C = history_C.history['val_accuracy'][-1]
final_train_acc_C = history_C.history['accuracy'][-1]

print('='*40)
print('모델 C 학습 완료!')
print(f'  파라미터 수      : 3,025개  (A의 {3025//37}배!)')
print(f'  최종 훈련 정확도  : {final_train_acc_C:.1%}')
print(f'  최종 검증 정확도  : {final_val_acc_C:.1%}')
print(f'  최종 손실        : {final_loss_C:.4f}')
print('='*40)
print(f'\n🔍 B vs C 검증 정확도 차이: {(final_val_acc_C - final_val_acc_B):+.1%}')

---
## ⭐ STEP 5 — 3개 손실 곡선 비교 + 최종 결과표

A · B · C 세 모델의 학습 곡선을 한 그래프에 겹쳐서 비교합니다.

**확인 포인트**
- 어느 모델이 가장 빠르게 손실이 내려가나요?
- 훈련 손실과 검증 손실 사이가 벌어지는 모델은?
- 최종 검증 정확도 순서가 예상과 같았나요?

In [ ]:
epochs_range = range(1, 101)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('모델 A · B · C 학습 곡선 비교', fontsize=15, fontweight='bold', y=1.02)

colors = {'A': '#3b82f6', 'B': '#14b8a6', 'C': '#f59e0b'}
labels = {'A': '모델 A (얕고 좁음)', 'B': '모델 B (기본형)', 'C': '모델 C (깊고 넓음)'}
histories = {'A': history_A, 'B': history_B, 'C': history_C}

# 왼쪽: 검증 손실 비교
ax = axes[0]
for key in ['A', 'B', 'C']:
    ax.plot(epochs_range, histories[key].history['val_loss'],
            label=labels[key], color=colors[key], linewidth=2.5)
ax.set_title('검증 손실 (Val Loss) 비교', fontsize=13)
ax.set_xlabel('에포크')
ax.set_ylabel('손실 (Loss)')
ax.legend()
ax.grid(True, alpha=0.3)

# 오른쪽: 검증 정확도 비교
ax = axes[1]
for key in ['A', 'B', 'C']:
    ax.plot(epochs_range,
            [v * 100 for v in histories[key].history['val_accuracy']],
            label=labels[key], color=colors[key], linewidth=2.5)
ax.set_title('검증 정확도 (Val Accuracy) 비교', fontsize=13)
ax.set_xlabel('에포크')
ax.set_ylabel('정확도 (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 과적합 진단 — 훈련 vs 검증 손실 비교
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('각 모델의 훈련 손실 vs 검증 손실 (과적합 진단)', fontsize=13, fontweight='bold')

for i, (key, color) in enumerate(colors.items()):
    ax = axes[i]
    ax.plot(epochs_range, histories[key].history['loss'],
            label='훈련 손실', color=color, linewidth=2.5)
    ax.plot(epochs_range, histories[key].history['val_loss'],
            label='검증 손실', color=color, linewidth=2, linestyle='--', alpha=0.7)
    ax.set_title(f'{labels[key]}', fontsize=12)
    ax.set_xlabel('에포크')
    ax.set_ylabel('손실')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('💡 두 선이 벌어지면 과적합 / 둘 다 높으면 과소적합 / 나란히 내려가면 정상')

In [ ]:
# 최종 성능 비교표
params = {'A': 37, 'B': 289, 'C': 3025}
struct = {
    'A': 'Dense(4)→Dense(1)',
    'B': 'Dense(16)→Dense(8)→Dense(1)',
    'C': 'Dense(64)→Dense(32)→Dense(16)→Dense(1)'
}
train_accs = {
    'A': final_train_acc_A,
    'B': final_train_acc_B,
    'C': final_train_acc_C
}
val_accs = {
    'A': final_val_acc_A,
    'B': final_val_acc_B,
    'C': final_val_acc_C
}

best_key = max(val_accs, key=val_accs.get)

print('\n' + '='*70)
print(f'{"모델":<8} {"구조":<40} {"파라미터":>8} {"훈련 정확도":>10} {"검증 정확도":>10}')
print('-'*70)
for key in ['A', 'B', 'C']:
    star = ' ⭐' if key == best_key else ''
    print(f'모델 {key}{star:<3} {struct[key]:<40} '
          f'{params[key]:>6}개   '
          f'{train_accs[key]:>8.1%}   '
          f'{val_accs[key]:>8.1%}')
print('='*70)
print(f'\n🏆 검증 정확도 1위: 모델 {best_key} ({val_accs[best_key]:.1%})')

# 과소/과적합 진단
print('\n📋 구조 진단:')
for key in ['A', 'B', 'C']:
    diff = train_accs[key] - val_accs[key]
    if val_accs[key] < 0.73:
        diag = '⚠️  과소적합 의심 — 모델 용량 부족'
    elif diff > 0.07:
        diag = '⚠️  과적합 의심 — 훈련/검증 차이가 큼'
    else:
        diag = '✅  균형 잡힌 학습'
    print(f'  모델 {key}: {diag}')

### 📝 결과 해석 포인트

| 현상 | 의미 | 해결 방법 |
|------|------|-----------|
| 훈련/검증 손실 **둘 다 높음** | 과소적합 — 모델 용량 부족 | 층·뉴런 수 늘리기 |
| 훈련 손실 낮고 검증 손실 **높음** | 과적합 — 훈련 데이터 외워버림 | 드롭아웃 추가, 데이터 늘리기 |
| 훈련/검증 손실 **나란히 내려감** | 정상 학습 ✅ | 현재 구조 유지 |

> 💡 **핵심 교훈**: 데이터 크기(768명)에 비해 모델이 너무 크면 오히려 성능이 떨어질 수 있습니다.  
> 더 큰 데이터셋(수십만 건)에서는 더 깊은 모델 C가 빛날 수 있습니다!

---
## ⭐ STEP 6 — 나만의 모델 설계

아래 코드에서 **층 수와 뉴런 수를 자유롭게 바꿔서** 모델 B보다 높은 검증 정확도를 달성해보세요!

**🎯 도전 목표**: `val_accuracy > 모델 B의 val_accuracy`

**힌트**:
- 층을 추가하거나 뉴런 수를 바꿔서 실험해보세요
- 너무 크게 키우면 과적합이 생길 수 있어요
- 에포크 수도 조정해볼 수 있어요 (50 ~ 200)

---
### ✏️ 아래 `TODO` 부분을 수정하세요!

In [ ]:
# ══════════════════════════════════════════════
# ✏️ TODO: 아래 부분을 자유롭게 수정하세요!
# ══════════════════════════════════════════════

MY_EPOCHS = 100  # ← 에포크 수 (50 ~ 200 사이에서 조정)

model_mine = Sequential([
    # ↓↓↓ 은닉층 구성을 직접 수정하세요 ↓↓↓
    Dense(16, activation='relu', input_shape=(8,)),  # 첫 번째 은닉층
    Dense(8,  activation='relu'),                    # 두 번째 은닉층
    # Dense(???, activation='relu'),                 # 세 번째 은닉층 (추가하려면 주석 해제)
    # ↑↑↑ 여기까지 수정 ↑↑↑
    Dense(1, activation='sigmoid')                   # 출력층 (수정 금지)
], name='my_model')

# ══════════════════════════════════════════════

model_mine.compile(
    optimizer=Adam(learning_rate=0.01),  # 학습률 고정
    loss='binary_crossentropy',          # 손실함수 고정
    metrics=['accuracy']
)

print('── 내 모델 구조 ──')
model_mine.summary()

In [ ]:
# 내 모델 학습
history_mine = model_mine.fit(
    X_train, y_train,
    epochs=MY_EPOCHS,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

final_val_acc_mine   = history_mine.history['val_accuracy'][-1]
final_train_acc_mine = history_mine.history['accuracy'][-1]
final_loss_mine      = history_mine.history['loss'][-1]

print('='*45)
print('내 모델 학습 완료!')
print(f'  훈련 정확도  : {final_train_acc_mine:.1%}')
print(f'  검증 정확도  : {final_val_acc_mine:.1%}')
print(f'  최종 손실    : {final_loss_mine:.4f}')
print('='*45)

# 모델 B와 비교
diff = final_val_acc_mine - final_val_acc_B
if diff > 0:
    print(f'\n🎉 성공! 모델 B보다 {diff:+.1%} 더 높습니다!')
    print('   다른 구조도 시도해서 더 높은 정확도에 도전해보세요!')
else:
    print(f'\n🤔 아직 모델 B({final_val_acc_B:.1%})가 더 좋습니다. ({diff:+.1%})')
    print('   층을 추가하거나 뉴런 수를 조정해서 다시 도전해보세요!')

In [ ]:
# 내 모델 vs 모델 B 손실 곡선 비교
mine_epochs_range = range(1, MY_EPOCHS + 1)
b_epochs_range = range(1, 101)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('내 모델 vs 모델 B (기준) 비교', fontsize=14, fontweight='bold')

# 손실 곡선
ax = axes[0]
ax.plot(mine_epochs_range, history_mine.history['loss'],
        label='내 모델 훈련', color='#22c55e', linewidth=2.5)
ax.plot(mine_epochs_range, history_mine.history['val_loss'],
        label='내 모델 검증', color='#22c55e', linewidth=2, linestyle='--', alpha=0.8)
ax.plot(b_epochs_range, history_B.history['val_loss'],
        label='모델 B 검증 (기준)', color='#14b8a6', linewidth=1.5, linestyle=':', alpha=0.7)
ax.set_title('손실 곡선 비교')
ax.set_xlabel('에포크')
ax.set_ylabel('손실')
ax.legend()
ax.grid(True, alpha=0.3)

# 정확도 곡선
ax = axes[1]
ax.plot(mine_epochs_range,
        [v * 100 for v in history_mine.history['val_accuracy']],
        label='내 모델 검증 정확도', color='#22c55e', linewidth=2.5)
ax.axhline(y=final_val_acc_B * 100, color='#14b8a6', linewidth=1.5,
           linestyle='--', label=f'모델 B 기준선 ({final_val_acc_B:.1%})')
ax.fill_between(mine_epochs_range,
                [v * 100 for v in history_mine.history['val_accuracy']],
                final_val_acc_B * 100,
                where=[v > final_val_acc_B for v in history_mine.history['val_accuracy']],
                alpha=0.15, color='#22c55e', label='B보다 높은 구간')
ax.set_title('검증 정확도 비교')
ax.set_xlabel('에포크')
ax.set_ylabel('정확도 (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 🎁 보너스 — 파라미터 수 자동 계산기

어떤 구조를 설계하더라도 파라미터 수를 자동으로 계산해줍니다.

In [ ]:
def calc_params(layer_sizes, input_size=8):
    """
    MLP 파라미터 수 계산기
    layer_sizes: 은닉층 뉴런 수 리스트 (출력층 제외)
    """
    total = 0
    prev = input_size
    for n in layer_sizes:
        params = prev * n + n  # 가중치 + 편향
        total += params
        print(f'  {prev}→{n}: {prev}×{n} + {n}(편향) = {params:,}개')
        prev = n
    # 출력층
    out_params = prev * 1 + 1
    total += out_params
    print(f'  {prev}→1 (출력): {prev}×1 + 1(편향) = {out_params:,}개')
    print(f'  {'─'*35}')
    print(f'  총 파라미터: {total:,}개')
    return total

print('== 모델 A ==')
calc_params([4])

print('\n== 모델 B ==')
calc_params([16, 8])

print('\n== 모델 C ==')
calc_params([64, 32, 16])

print('\n== 내가 만든 구조 예시 ==')
calc_params([32, 16, 8])  # ← 원하는 구조로 변경 가능

---
## ✅ 오늘 배운 것 — 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| **층 수 (Depth)** | 깊을수록 추상화 능력 증가 — 하지만 기울기 소실·과적합 위험 |
| **뉴런 수 (Width)** | 많을수록 표현력 증가 — 하지만 파라미터 폭발·과적합 위험 |
| **활성화 함수** | 없으면 층을 쌓아도 선형 회귀와 동일! 은닉층: ReLU / 출력층: Sigmoid |
| **과소적합** | 훈련·검증 손실 둘 다 높음 → 모델이 너무 단순 |
| **과적합** | 훈련 손실↓ vs 검증 손실↑ → 훈련 데이터를 외워버림 |
| **변인 통제** | 구조만 바꾸고 나머지 조건 동일하게 유지해야 공정한 비교 |

> **🔑 핵심 교훈**: "층이 많다고 항상 좋은 게 아니다.  
> 데이터 크기와 복잡도에 맞는 구조를 실험으로 찾아야 한다!"

---
### 🚀 11주차 예고
다음 주차에서는 모델이 틀리는 방식을 자세히 진단하고 → **여러 클래스를 동시에 분류**하는 다중 분류 문제를 다룹니다!